<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# DXQuant · Derivatives Analytics with Python
## Notebook 2 · Monte Carlo Pricing and Greeks

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/dxquant/blob/main/notebooks/02_dxquant_monte_carlo_and_greeks.ipynb)


## Notebook Goals

This notebook uses `dxquant.monte_carlo.MonteCarloPricer` to price terminal and path-dependent
payoffs. You will

- set up a Monte Carlo pricer under Geometric Brownian Motion (GBM),
- price a European call and compute delta and gamma by finite differences,
- price two path-dependent contracts (Asian and barrier), and
- extract simple risk statistics from the payoff distribution.


Sampling errors use independent paths or antithetic pair means. A single moment-matched batch reports `NaN` for its error; use independent seeded replications to measure batch dispersion. A shared `scenario_set` explicitly asserts ordered joint scenarios for portfolio payoff risk; it does not construct correlations.


## Setup

In Google Colab, select **Runtime → Run all** with a standard CPU runtime. This cell installs DXQuant and Matplotlib if they are missing. No GPU, Google Drive mount, or API credentials are needed. The examples use synthetic inputs.

Before the first PyPI release, installation uses the public GitHub `main` branch. When running from a local checkout, the setup cell installs the local package instead.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = []
if importlib.util.find_spec("dxquant") is None:
    package_source = (
        "dxquant @ git+https://github.com/yhilpisch/dxquant.git@main"
    )
    for directory in (Path.cwd(), Path.cwd().parent):
        local_package = directory
        if (local_package / "pyproject.toml").is_file():
            package_source = str(local_package.resolve())
            break
    requirements.append(package_source)
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.8")
if requirements:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *requirements]
    )


In [ ]:
import datetime as dt

import numpy as np

import dxquant as dx

np.set_printoptions(precision=6, suppress=True)

## Model and Pricing Setup

We use a constant short rate for discounting and a GBM for the underlying dynamics. A fixed
random seed makes the Monte Carlo estimate deterministic for tutorial purposes.

In [ ]:
pricing_date = dt.date(2024, 1, 1)
maturity = dt.date(2025, 1, 1)

spot = 100.0
strike = 105.0
rate = 0.02
vol = 0.20

env = dx.MarketEnvironment(name="mc", pricing_date=pricing_date)
env.add_curve("discount", dx.ConstantShortRateCurve("disc", pricing_date, rate))

process = dx.GeometricBrownianMotion(
    drift=rate,
    volatility=vol,
    seed=123,
    moment_matching=False,
)
pricer = dx.MonteCarloPricer(
    env=env,
    process=process,
    discount_curve_key="discount",
    paths=40000,
    steps=96,
)

pricer

## European Call: Monte Carlo Price, Delta, Gamma

Delta and gamma are computed by finite differences using repeated re-valuations.

In [ ]:
call = dx.EuropeanCallPayoff(strike=strike)
call_result = pricer.price_terminal(call, maturity=maturity, spot=spot)

delta = pricer.sensitivity(call, maturity=maturity, spot=spot, order=1)
gamma = pricer.sensitivity(call, maturity=maturity, spot=spot, order=2)

print(f"European call price: {call_result.price:.6f}")
print(f"Std. error:          {call_result.std_error:.6f}")
print(f"Delta:              {delta:.6f}")
print(f"Gamma:              {gamma:.6f}")

## Two Path-Dependent Payoffs

DXQuant separates terminal payoffs (depend only on the final state) from path-dependent payoffs
(depend on the full simulated path).

In [ ]:
asian = dx.ArithmeticAsianCallPayoff(strike=strike)
asian_result = pricer.price_path_dependent(asian, maturity=maturity, spot=spot)

barrier = dx.UpAndOutBarrierCallPayoff(strike=strike, barrier=130.0)
barrier_result = pricer.price_path_dependent(
    barrier, maturity=maturity, spot=spot
)

print(f"Asian call price:    {asian_result.price:.6f}")
print(f"Barrier call price:  {barrier_result.price:.6f}")

## Distribution-Based Risk Statistics

Monte Carlo produces a payoff distribution. Simple quantiles are often useful for sanity checks
and risk reporting.

In [ ]:
p05 = call_result.percentile(5.0)
p50 = call_result.percentile(50.0)
p95 = call_result.percentile(95.0)

print(f"5% percentile:       {p05:.6f}")
print(f"50% percentile:      {p50:.6f}")
print(f"95% percentile:      {p95:.6f}")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">